# Yield window by GPQR

In [ ]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

sys.path.insert(0, os.path.abspath(".."))

warnings.filterwarnings("ignore")

## Load data

In [ ]:
X = pd.read_csv("../_temp/X.csv")
y = pd.read_csv("../_temp/y.csv")
Xpred = pd.read_csv("../_temp/Xpred_2D.csv", index_col=[0, 1, 2, 3])
Delaunay = np.load("../_temp/delaunay.Xpred_2D.npy")

H_priormean = np.load("../_temp/H.prior_mean.Xpred_2D.npy")
phi_priormean = np.load("../_temp/phi.prior_mean.Xpred_2D.npy")

H_median = np.load("../_temp/H.quantiles.Xpred_2D.npy")[..., 2]
phi_median = np.load("../_temp/phi.quantiles.Xpred_2D.npy")[..., 2]

H_marginal = np.load("../_temp/H.marginal.Xpred_2D.npy")
phi_marginal = np.load("../_temp/phi.marginal.Xpred_2D.npy")

joint = np.load("../_temp/joint_probability.Xpred_2D.npy")

## Plot

In [ ]:
X_Slurry = X["Slurry"]
Xpred_Slurry = Xpred.index.get_level_values("Slurry")

Slurries = X_Slurry.unique()

## Prior mean (H)

Black lines indicate interpolation region.

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray"), mcolors.to_rgba("tab:blue")],
    N=N_COLORS,
)

levels = np.linspace(H_priormean.min(), H_priormean.max(), N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey=True)

for slurry, ax in zip(Slurries, axes):
    X_idx = X_Slurry == slurry
    Xpred_idx = Xpred_Slurry == slurry

    x = X[X_idx].drop(columns="Slurry").to_xarray().to_array().values
    xpred = Xpred[Xpred_idx].droplevel("Slurry").to_xarray().to_array().values
    Rgt = xpred[0]
    Ca = xpred[1]

    delaunay = Delaunay[Xpred_idx].reshape(xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        H_priormean[Xpred_idx].reshape(xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.scatter(
        x[0, ...],
        x[1, ...],
        c=y[X_idx]["H"],
        cmap=cmap,
        norm=norm,
        edgecolor="k",
        linewidth=0.5,
        s=20,
    )

    ax.contour(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Prior mean (H)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Prior mean (phi)

Black lines indicate interpolation region.

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray"), mcolors.to_rgba("tab:blue")],
    N=N_COLORS,
)

levels = np.linspace(phi_priormean.min(), phi_priormean.max(), N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey=True)

for slurry, ax in zip(Slurries, axes):
    X_idx = X_Slurry == slurry
    Xpred_idx = Xpred_Slurry == slurry

    x = X[X_idx].drop(columns="Slurry").to_xarray().to_array().values
    xpred = Xpred[Xpred_idx].droplevel("Slurry").to_xarray().to_array().values
    Rgt = xpred[0]
    Ca = xpred[1]

    delaunay = Delaunay[Xpred_idx].reshape(xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        phi_priormean[Xpred_idx].reshape(xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.scatter(
        x[0, ...],
        x[1, ...],
        c=y[X_idx]["phi"],
        cmap=cmap,
        norm=norm,
        edgecolor="k",
        linewidth=0.5,
        s=20,
    )

    ax.contour(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Prior mean (phi)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Posterior mean of median (H)

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray"), mcolors.to_rgba("tab:blue")],
    N=N_COLORS,
)

levels = np.linspace(H_median.min(), H_median.max(), N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey=True)

for slurry, ax in zip(Slurries, axes):
    X_idx = X_Slurry == slurry
    Xpred_idx = Xpred_Slurry == slurry

    x = X[X_idx].drop(columns="Slurry").to_xarray().to_array().values
    xpred = Xpred[Xpred_idx].droplevel("Slurry").to_xarray().to_array().values
    Rgt = xpred[0]
    Ca = xpred[1]

    delaunay = Delaunay[Xpred_idx].reshape(xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        H_median[Xpred_idx].reshape(xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.scatter(
        x[0, ...],
        x[1, ...],
        c=y[X_idx]["H"],
        cmap=cmap,
        norm=norm,
        edgecolor="k",
        linewidth=0.5,
        s=20,
    )

    ax.contour(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Posterior mean of median (H)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Posterior mean of median (phi)

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray"), mcolors.to_rgba("tab:blue")],
    N=N_COLORS,
)

levels = np.linspace(phi_median.min(), phi_median.max(), N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey=True)

for slurry, ax in zip(Slurries, axes):
    X_idx = X_Slurry == slurry
    Xpred_idx = Xpred_Slurry == slurry

    x = X[X_idx].drop(columns="Slurry").to_xarray().to_array().values
    xpred = Xpred[Xpred_idx].droplevel("Slurry").to_xarray().to_array().values
    Rgt = xpred[0]
    Ca = xpred[1]

    delaunay = Delaunay[Xpred_idx].reshape(xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        phi_median[Xpred_idx].reshape(xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.scatter(
        x[0, ...],
        x[1, ...],
        c=y[X_idx]["phi"],
        cmap=cmap,
        norm=norm,
        edgecolor="k",
        linewidth=0.5,
        s=20,
    )

    ax.contour(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Posterior mean of median (phi)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Marginal yield (H)

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray", alpha=0.3), mcolors.to_rgba("tab:blue", alpha=0.8)],
    N=N_COLORS,
)

levels = np.linspace(0, 1, N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey=True)

for slurry, ax in zip(Slurries, axes):
    Xpred_idx = Xpred_Slurry == slurry

    xpred = Xpred[Xpred_idx].droplevel("Slurry").to_xarray().to_array().values
    Rgt = xpred[0]
    Ca = xpred[1]

    delaunay = Delaunay[Xpred_idx].reshape(xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        H_marginal[Xpred_idx].reshape(xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Marginal yield (H)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Marginal yield (phi)

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray", alpha=0.3), mcolors.to_rgba("tab:blue", alpha=0.8)],
    N=N_COLORS,
)

levels = np.linspace(0, 1, N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey=True)

for slurry, ax in zip(Slurries, axes):
    Xpred_idx = Xpred_Slurry == slurry

    xpred = Xpred[Xpred_idx].droplevel("Slurry").to_xarray().to_array().values
    Rgt = xpred[0]
    Ca = xpred[1]

    delaunay = Delaunay[Xpred_idx].reshape(xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        phi_marginal[Xpred_idx].reshape(xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Marginal yield (phi)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()

## Joint yield

In [ ]:
N_COLORS = 8
cmap = mcolors.LinearSegmentedColormap.from_list(
    "gray_blue",
    [mcolors.to_rgba("gray", alpha=0.3), mcolors.to_rgba("tab:blue", alpha=0.8)],
    N=N_COLORS,
)

levels = np.linspace(0, 1, N_COLORS + 1)
norm = mcolors.BoundaryNorm(levels, ncolors=N_COLORS)

In [ ]:
fig, axes = plt.subplots(1, len(Slurries), sharex=True, sharey=True)

for slurry, ax in zip(Slurries, axes):
    Xpred_idx = Xpred_Slurry == slurry

    xpred = Xpred[Xpred_idx].droplevel("Slurry").to_xarray().to_array().values
    Rgt = xpred[0]
    Ca = xpred[1]

    delaunay = Delaunay[Xpred_idx].reshape(xpred.shape[1:])
    delaunay_masked = np.full_like(delaunay, np.nan, dtype=float)

    ax.contourf(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        joint[Xpred_idx].reshape(xpred.shape[1:]).squeeze(axis=-1),
        cmap=cmap,
        norm=norm,
        levels=levels,
    )

    ax.contour(
        xpred[0, ...].squeeze(axis=-1),
        xpred[1, ...].squeeze(axis=-1),
        delaunay.squeeze(axis=-1).astype(float),
        levels=[0.5],
        colors="k",
    )

    ax.set_title(slurry)

fig.tight_layout(rect=[0.02, 0.05, 1.0, 0.8])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.18, 0.82, 0.64, 0.04])
cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Joint yield (H & phi)", labelpad=6)
cbar.ax.xaxis.set_ticks_position("top")
cbar.ax.xaxis.set_label_position("top")
cbar.ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
cbar.ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:.2f}"))

fig.supxlabel("Rgt")
fig.supylabel("Ca")

fig.show()